In [ ]:
# MATLAB section 1/11 for ValidationDataSet: Software Validation Data Set

# % Software Validation Data Set
# The purpose of this example is to two important test cases of data to
# validate the Neural Spike Analysis Toolbox.
#
#

# Python translation bootstrap for this helpfile.

# Topic: ValidationDataSet
# Execution group: full
# Workflow family: data
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/ValidationDataSet.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "ValidationDataSet"
FAMILY = "data"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"ValidationDataSet: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"ValidationDataSet: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"ValidationDataSet: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"ValidationDataSet: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB section 2/11 for ValidationDataSet: Case #1: Constant Rate Poisson Process

# % Case #1: Constant Rate Poisson Process
# First we want to show that when neural firing activity is generated from
# a constant rate poisson process, the algorithm is able to estimate the
# value of this constant rate.
#
# MATLAB: clear all;
# MATLAB: close all;
#
# MATLAB: p=0.01;         % bernoilli probability
# MATLAB: N=100001;       % Number of coin flips
# MATLAB: delta = 0.001;  % binsize
# MATLAB: T=N*delta;      % total time window
# MATLAB: lambda=N*p/T    % lambda*T = N*p
#
# MATLAB: mu = log(lambda*delta/(1-lambda*delta))

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/11 for ValidationDataSet: Section

# %
# Now generate data for two neurons based on this constant rate
# MATLAB: for i=1:2
# MATLAB:     t=linspace(0,T,N);
# MATLAB:     ind=rand(1,N)<p;     %generate the coin-flip indices for heads or 1's
# MATLAB:     spikeTimes = t(ind); %get time spikes based on indices
# MATLAB:     nst{i}=nspikeTrain(spikeTimes,'',delta); % create neuron spike train
# MATLAB:     nst{i}.setMinTime(0);
# MATLAB:     nst{i}.setMaxTime(T);
# MATLAB: end
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/11 for ValidationDataSet: Section

# %
# For a sanity check we can plot the ISI histogram for the two neurons and
# verify that they are exponentially distributed with \lambda = N*p/T;
# MATLAB: nst{1}.plotISIHistogram;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 4

_ = section_index


In [ ]:
# MATLAB section 5/11 for ValidationDataSet: Section

# %
# Setup the analysis using the Neural Spike Analysis Toolbox
# Since we are going to try to fit a constant rate model, we create a
# baseline covariate that is constant and equal to 1 for the duration of
# the trial. This data in the covarate will be labeled 'constant';
#
# MATLAB: spikeColl=nstColl(nst); %create a nstColl - a collection of spikeTrains
# MATLAB: cov=Covariate(t,ones(length(t),1),'Baseline','s','','',{'mu'});
# MATLAB: cc=CovColl({cov}); % Gather all the covariates
# MATLAB: trial=Trial(spikeColl, cc); %Create the trial
#
# Specify how we want to perform the analysis
# MATLAB: clear c;
# MATLAB: sampleRate=1000;
# Try just using the 'constant' data from the baseline covariate
# MATLAB: c{1} = TrialConfig({{'Baseline','mu'}},sampleRate,[],[]);
# MATLAB: c{1}.setName('Baseline');
# MATLAB: cfgColl= ConfigColl(c); %place desired configurations in a ConfigColl structure
#
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 5

_ = section_index


In [ ]:
# MATLAB section 6/11 for ValidationDataSet: Section

# %
# Run the analysis
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);
# MATLAB: results{1}.plotResults; subplot(2,4,[5 6]); plot(mu,'ro', 'MarkerSize',10);
# MATLAB: results{2}.plotResults; subplot(2,4,[5 6]); plot(mu,'ro', 'MarkerSize',10);
# MATLAB: figure;
# MATLAB: subplot(1,2,1);results{1}.lambda.plot; hold on; plot(results{1}.lambda.time,lambda*ones(length(results{1}.lambda.time),1),'r-.','LineWidth',3);
# MATLAB: subplot(1,2,2);results{2}.lambda.plot; hold on; plot(results{2}.lambda.time,lambda*ones(length(results{2}.lambda.time),1),'r-.','LineWidth',3);

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 6

_ = section_index


In [ ]:
# MATLAB section 7/11 for ValidationDataSet: Case #2: Piece-wise Constant Rate Poisson Process

# % Case #2: Piece-wise Constant Rate Poisson Process
# Make a joint process be the sum of two independet and non-overlapping
# Poisson processes with different rates. During the first interval, only
# observer arrivals from process 1, and during the second interval only
# observe arrivals from the second process. Compare the results of estimate
# the complete process as the sum of two distinct independent and
# non-overlapping Poisson processes versus a single constant rate process.
#
# Process 1
# MATLAB: p1=0.001; % bernoilli probability of process 1
# MATLAB: N1=100000; %
# MATLAB: delta = 0.001;
# MATLAB: T1=N1*delta;
# MATLAB: lambda1=N1*p1/T1    % lambda*T = N*p
# MATLAB: mu1 = log(lambda1*delta/(1-lambda1*delta))
# Process 2
# MATLAB: p2=0.01;  % bernoilli probability of process 1
# MATLAB: N2=100000;
# MATLAB: T2=N2*delta;
# MATLAB: lambda2=N2*p2/T2    % lambda*T = N*p
# MATLAB: mu2 = log(lambda2*delta/(1-lambda2*delta))
#
# Estimate of constant rate process:
# MATLAB: lambdaConst = (N1*p1 + N2*p2)/(T1+T2)
# MATLAB: muConst = log(lambdaConst*delta/(1-lambdaConst*delta))
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 7

_ = section_index


In [ ]:
# MATLAB section 8/11 for ValidationDataSet: Section

# %
# Generate the data for 2 neurons
# MATLAB: for i=1:2
# MATLAB:     tTot = linspace(0,(T1+T2),(N1+N2+1));
# MATLAB:     t1=tTot(tTot<=T1);
# MATLAB:     ind1=rand(1,N1)<p1;
# MATLAB:     spikeTimes1 = t1(ind1);
# MATLAB:     t2=tTot(tTot>T1);
# MATLAB:     ind2=rand(1,N2)<p2;
# MATLAB:     spikeTimes2 = t2(ind2);
# MATLAB:     tTot = [t1'; t2'];
#
# MATLAB:     nst{i}=nspikeTrain([spikeTimes1 spikeTimes2],'',delta);
# MATLAB:     nst{i}.setMinTime(0);
# MATLAB:     nst{i}.setMaxTime(max(t2));
# MATLAB: end
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 8

_ = section_index


In [ ]:
# MATLAB section 9/11 for ValidationDataSet: Section

# %
# Generate the trial data;
# MATLAB: spikeColl=nstColl(nst); %create a nstColl
# MATLAB: cov=Covariate(tTot,[ones(length(tTot),1), tTot<=max(t1), tTot>max(t1)],'Baseline','s','','',{'muConst','mu1','mu2'});
# MATLAB: cc=CovColl({cov});
#
# Specify how we want to perform the analysis
# MATLAB: sampleRate=1000;
# MATLAB: trial=Trial(spikeColl, cc);
# MATLAB: clear c;
# Constant rate throughout
# MATLAB: c{1} = TrialConfig({{'Baseline','muConst'}},sampleRate,[],[]);
# MATLAB: c{1}.setName('Baseline');
# Constant rate for epoch1 and Constat rate for epoch2 but distinct
# MATLAB: c{2} = TrialConfig({{'Baseline','mu1','mu2'}},sampleRate,[],[]);
# MATLAB: c{2}.setName('Variable');
# MATLAB: cfgColl= ConfigColl(c);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 9

_ = section_index


In [ ]:
# MATLAB section 10/11 for ValidationDataSet: Section

# %
# Run the analysis
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);
# MATLAB: results{1}.plotResults;
# MATLAB: results{2}.plotResults;
# MATLAB: figure;
# MATLAB: subplot(1,2,1); results{1}.lambda.plot;
# MATLAB: subplot(1,2,2); results{2}.lambda.plot;

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 10

_ = section_index


In [ ]:
# MATLAB section 11/11 for ValidationDataSet: Section

# %
# Compare the results across the two neurons
# MATLAB: Summary = FitResSummary(results);
# MATLAB: Summary.plotSummary;
#
#
#

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "clear all;",
    "close all;",
    "p=0.01;         % bernoilli probability",
    "N=100001;       % Number of coin flips",
    "delta = 0.001;  % binsize",
    "T=N*delta;      % total time window",
    "lambda=N*p/T    % lambda*T = N*p",
    "mu = log(lambda*delta/(1-lambda*delta))",
    "for i=1:2",
    "t=linspace(0,T,N);",
    "ind=rand(1,N)<p;     %generate the coin-flip indices for heads or 1's",
    "spikeTimes = t(ind); %get time spikes based on indices",
    "nst{i}=nspikeTrain(spikeTimes,'',delta); % create neuron spike train",
    "nst{i}.setMinTime(0);",
    "nst{i}.setMaxTime(T);",
    "end",
    "nst{1}.plotISIHistogram;",
    "spikeColl=nstColl(nst); %create a nstColl - a collection of spikeTrains",
    "cov=Covariate(t,ones(length(t),1),'Baseline','s','','',{'mu'});",
    "cc=CovColl({cov}); % Gather all the covariates",
    "trial=Trial(spikeColl, cc); %Create the trial",
    "clear c;",
    "sampleRate=1000;",
    "c{1} = TrialConfig({{'Baseline','mu'}},sampleRate,[],[]);",
    "c{1}.setName('Baseline');",
    "cfgColl= ConfigColl(c); %place desired configurations in a ConfigColl structure",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "results{1}.plotResults; subplot(2,4,[5 6]); plot(mu,'ro', 'MarkerSize',10);",
    "results{2}.plotResults; subplot(2,4,[5 6]); plot(mu,'ro', 'MarkerSize',10);",
    "figure;",
    "subplot(1,2,1);results{1}.lambda.plot; hold on; plot(results{1}.lambda.time,lambda*ones(length(results{1}.lambda.time),1),'r-.','LineWidth',3);",
    "subplot(1,2,2);results{2}.lambda.plot; hold on; plot(results{2}.lambda.time,lambda*ones(length(results{2}.lambda.time),1),'r-.','LineWidth',3);",
    "p1=0.001; % bernoilli probability of process 1",
    "N1=100000; %",
    "delta = 0.001;",
    "T1=N1*delta;",
    "lambda1=N1*p1/T1    % lambda*T = N*p",
    "mu1 = log(lambda1*delta/(1-lambda1*delta))",
    "p2=0.01;  % bernoilli probability of process 1",
    "N2=100000;",
    "T2=N2*delta;",
    "lambda2=N2*p2/T2    % lambda*T = N*p",
    "mu2 = log(lambda2*delta/(1-lambda2*delta))",
    "lambdaConst = (N1*p1 + N2*p2)/(T1+T2)",
    "muConst = log(lambdaConst*delta/(1-lambdaConst*delta))",
    "for i=1:2",
    "tTot = linspace(0,(T1+T2),(N1+N2+1));",
    "t1=tTot(tTot<=T1);",
    "ind1=rand(1,N1)<p1;",
    "spikeTimes1 = t1(ind1);",
    "t2=tTot(tTot>T1);",
    "ind2=rand(1,N2)<p2;",
    "spikeTimes2 = t2(ind2);",
    "tTot = [t1'; t2'];",
    "nst{i}=nspikeTrain([spikeTimes1 spikeTimes2],'',delta);",
    "nst{i}.setMinTime(0);",
    "nst{i}.setMaxTime(max(t2));",
    "end",
    "spikeColl=nstColl(nst); %create a nstColl",
    "cov=Covariate(tTot,[ones(length(tTot),1), tTot<=max(t1), tTot>max(t1)],'Baseline','s','','',{'muConst','mu1','mu2'});",
    "cc=CovColl({cov});",
    "sampleRate=1000;",
    "trial=Trial(spikeColl, cc);",
    "clear c;",
    "c{1} = TrialConfig({{'Baseline','muConst'}},sampleRate,[],[]);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','mu1','mu2'}},sampleRate,[],[]);",
    "c{2}.setName('Variable');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "results{1}.plotResults;",
    "results{2}.plotResults;",
    "figure;",
    "subplot(1,2,1); results{1}.lambda.plot;",
    "subplot(1,2,2); results{2}.lambda.plot;",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for ValidationDataSet.")

# ValidationDataSet: load MATLAB-gold trial matrix and reproduce raster/PSTH/significance summaries.
from pathlib import Path
import nstat
from scipy.io import loadmat
fixture_path = Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/ValidationDataSet_gold.mat"
m = loadmat(str(fixture_path), squeeze_me=True, struct_as_record=False)
dt = float(np.asarray(m["dt_val"], dtype=float).reshape(-1)[0]); time = np.asarray(m["time_val"], dtype=float).reshape(-1)
trial_matrix = np.asarray(m["trial_matrix_val"], dtype=float); psth = np.asarray(m["psth_val"], dtype=float).reshape(-1); sem = np.asarray(m["sem_val"], dtype=float).reshape(-1)
rates, prob_mat, sig_mat = DecodingAlgorithms.compute_spike_rate_cis(spike_matrix=trial_matrix, alpha=0.05)
exp_rates = np.asarray(m["expected_rate_val"], dtype=float).reshape(-1); exp_prob = np.asarray(m["expected_prob_val"], dtype=float); exp_sig = np.asarray(m["expected_sig_val"], dtype=int)
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=False)
for k in range(min(18, trial_matrix.shape[0])): axes[0].vlines(time[trial_matrix[k] > 0], k + 0.6, k + 1.4, linewidth=0.5)
axes[0].set_title(f"{TOPIC}: trial raster"); axes[0].set_ylabel("trial")
axes[1].plot(time, psth, color="tab:blue", linewidth=1.2); axes[1].fill_between(time, psth - sem, psth + sem, color="tab:blue", alpha=0.2); axes[1].set_ylabel("Hz"); axes[1].set_title("PSTH mean +/- SEM")
im = axes[2].imshow(prob_mat, aspect="auto", origin="lower", cmap="viridis"); axes[2].set_title("Trial-by-trial spike-rate p-values"); axes[2].set_xlabel("trial"); axes[2].set_ylabel("trial"); fig.colorbar(im, ax=axes[2], fraction=0.03, pad=0.02)
plt.tight_layout(); plt.show()
rate_err = float(np.max(np.abs(rates - exp_rates))); prob_err = float(np.max(np.abs(prob_mat - exp_prob))); sig_mismatch = float(np.count_nonzero(sig_mat != exp_sig))
assert rate_err <= 1e-10 and prob_err <= 1e-10 and sig_mismatch == 0.0
CHECKPOINT_METRICS = {"rate_max_abs_error": rate_err, "prob_max_abs_error": prob_err, "sig_mismatch_count": sig_mismatch}
CHECKPOINT_LIMITS = {"rate_max_abs_error": (0.0, 1e-10), "prob_max_abs_error": (0.0, 1e-10), "sig_mismatch_count": (0.0, 0.0)}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
